<a href="https://colab.research.google.com/github/ValentinaZubareva2906/make_AI_product/blob/main/prompting/2_2_resume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install langchain langchain-classic langchain-openai openai tiktoken -q

In [ ]:
# Если используете ключ из курса, запустите эту ячейку
from langchain_openai import ChatOpenAI

course_api_key= "sk-CTWDfcT-MqN2gUhZh_3qbA"
#course_api_key = getpass(prompt='Введите API-ключ полученный в боте:')

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")

In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
df = pd.read_csv('https://stepik.org/media/attachments/lesson/1110806/vacancies_messages_50.csv')

In [ ]:
#df.head()

In [ ]:
from langchain_classic import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

In [ ]:
# Понадобится ещё одна сущность: схема ответа - ResponseSchema
job_title_schema = ResponseSchema(
    name="job_title",
    description="Как указано в описании вакансии, на том же языке. Проверяй полное совпадение, без учета регистра. "
                "Если написан грейд, его нужно убрать (например, Senior Python developer -> Python developer, C++ разработчик (middle, senior) ->C++ разработчик)"
                "Если информация явно не определена, поставь значение None."
)

company_schema = ResponseSchema(
    name="company",
    description="Как указано в описании вакансии, на том же языке. Проверяй полное совпадение, без учета регистра. "
                "Указывай только название (не пиши финтех, крупная компания, мобильная игра и т.д.)"
                "Если информация явно не определена, поставь значение None."
)

salary_schema = ResponseSchema(
    name="salary",
    description="Числа пиши без пробелов, не пиши тыс. или к.(умножай в таком случае на 1000), не пиши фикс, плюшки, премии, % от продаж, техника и т.д."
                "Не пиши net, gross, на руки и т.д."
                "После числа (диапазона чисел) указывай валюту руб. или $ после пробела."
                "Если указан диапазон, то пиши его через тире, около тире пробелы не ставь (например, 100к-150к рублей -> 100000-150000 руб.)."
                "Если указана только нижняя граница (от 2000$ или 100000+руб), то пиши это значение, используя слово от (например, 100к+руб -> от 100000 руб.)."
                "Если указана только верхняя граница, то пиши это значение, используя слово до (например, до 100к руб -> до 100000 руб.)."
                "Если зарплата указана за час, то в конце добавляй в час."
                "Если информация явно не определена, поставь значение None."
)

tg_schema = ResponseSchema(
    name="tg",
    description="Указывай контакт в телеграмм, используя @. Проверяй полное совпадение, с учетом регистра. "
                "Если указано несколько контаков, то указывай их через запятую (не забывая про пробел после запятой)."
                "Если информация явно не определена, поставь значение None."
)

grade_schema = ResponseSchema(
    name="grade",
    description="Возможные значения intern, junior, junior+, middle, middle+, senior, lead."
                "Если указано несколько значений, то пиши их через запятую в порядке возрастания."
                "Если информация явно не определена, поставь значение None."
)

response_schemas = [
    job_title_schema,
    company_schema,
    salary_schema,
    tg_schema,
    grade_schema
]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
review_template = """\
Из следующего текста извлеки информацию:

job_title: Извлеки название вакансии.
Если не можешь найти - напиши None.

company: Извлеки название компании.
Если не можешь найти - напиши None.

tg: Извлеки контакт в телеграмм.
Если не можешь найти - напиши None.

grade: Извлеки возможные значения intern, junior, junior+, middle, middle+, senior, lead.
Если не можешь найти - напиши None.


text: {text}

{format_instructions}

"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template=review_template)

In [ ]:
dict_list = []

In [ ]:
for text in df['text']:
  messages = prompt.format_messages(text = text, format_instructions = format_instructions)
  response = llm.invoke(messages[0].content)
  parsed_output = output_parser.parse(response.content)
  dict_list.append(parsed_output)

In [ ]:
dict_list = []
for text in df['text']:
    try:
        messages = prompt.format_messages(text=text, format_instructions=format_instructions)
        if not messages:
            print(f"Пустое сообщение для текста: {text}")
            continue
        response = llm.invoke(messages[0].content)
        parsed_output = output_parser.parse(str(response.content))
        dict_list.append(parsed_output)
    except Exception as e:
        print(f"Ошибка при обработке текста '{text[:50]}...': {e}")

In [ ]:
df_parsed = pd.DataFrame(dict_list)
df_parsed.reset_index(drop=True, inplace=True)
#df_results


In [ ]:
df_other = pd.DataFrame(df)
df_other.reset_index(drop=True, inplace=True)

In [ ]:
df_combined = pd.concat([df_other, df_parsed], axis=1)

In [ ]:
df_combined.to_csv('2_2_2_resume.csv', index=False)